# Extract data from CERCA raw files

In [1]:
from docx import Document
import pandas as pd
from google.cloud import bigquery
import requests
from tqdm import tqdm
import time
from df2gspread import gspread2df as g2d

In [2]:
interest_centers = ['BETA', 'CREAF', 'ICN2', 'ISGlobal', 'ResearchMar']
file_path = '../data/external/5_Bibliometria_SIRIS_081025/'

## Extract DOI list

**Each file has a different format so we go center by center**

### BETA

In [3]:
center_name = 'BETA/'
file_name = 'LIST OF SCIENTIFIC PUBLICATIONS_BETA'

In [4]:
doc = Document(file_path + 'BM_' + center_name + file_name + '.docx')
pubs = [para.text for para in doc.paragraphs]
DOI_tmp = [pub.split('DOI:', 1)[1].strip() for pub in pubs  if 'DOI:' in pub]
DOI_BETA = [(DOI.split('https://doi.org/', 1)[1].strip() if 'https://doi.org/' in DOI else DOI) for DOI in DOI_tmp]
df_BETA = pd.DataFrame(DOI_BETA, columns = ['DOI'])
df_BETA['Center'] = 'BETA'
df_BETA

,DOI,Center
0,10.1016/j.scitotenv.2023.168824,BETA
1,10.23818/limn.43.07,BETA
2,10.1016/j.aquatox.2024.106843,BETA
3,10.3390/agronomy14050935,BETA
4,10.32347/2077-3455.2024.68.215-227,BETA
...,...,...
63,10.1016/j.bcab.2021.102114,BETA
64,10.1016/j.sajb.2021.06.035,BETA
65,10.1002/ecy.3614,BETA
66,10.3390/ijms222011277,BETA


### CREAF

In [5]:
center_name = 'CREAF/'
file_name = 'Publicacions CREAF 2021-2024'

In [6]:
df = pd.read_excel(file_path + 'BM_' + center_name + file_name + '.xlsx', skiprows = 5)
df_CREAF = df[['doi']].rename(columns = {'doi': 'DOI'})
df_CREAF['Center'] = 'CREAF'
df_CREAF

,DOI,Center
0,10.1038/s43705-021-00073-5,CREAF
1,10.7325/galemys.2022.n5,CREAF
2,10.1109/IGARSS53475.2024.10640524,CREAF
3,10.1109/IGARSS53475.2024.10641964,CREAF
4,10.1109/MetroAgriFor63043.2024.10948835,CREAF
...,...,...
928,10.18601/01245996.v24n47.12,CREAF
929,10.1038/s43247-021-00229-0,CREAF
930,10.34133/2022/9764982,CREAF
931,10.34133/remotesensing.0085,CREAF


### ICN2

In [7]:
center_name = 'ICN2/'
file_name = 'ICN2_DOIs_Pubs2021-2024_CERCA2025'

In [8]:
df_ICN2 = pd.read_excel(file_path + 'BM_' + center_name + file_name + '.xlsx').rename(columns = {'Doi': 'DOI'})
df_ICN2['Center'] = 'ICN2'
df_ICN2

,DOI,Center
0,10.59277/ROMJIST.2024.2.09,ICN2
1,10.1002/ece2.12,ICN2
2,10.1093/mam/ozae044.520,ICN2
3,10.1103/physrevlett.132.266301,ICN2
4,10.7203/metode.15.27225,ICN2
...,...,...
823,10.1103/PhysRevB.108.054524,ICN2
824,10.1021/acsaem.1c02919,ICN2
825,10.1016/j.synthmet.2021.116844,ICN2
826,10.1038/s42254-021-00318-1,ICN2


### ISGlobal

In [14]:
center_name = 'ISGlobal/'
file_name = 'ISGlobal_DOIs Publications_2021-2024'

In [15]:
df = pd.read_excel(file_path + 'BM_' + center_name + file_name + '.xlsx', skiprows = 1)
df_ISGlobal = df[['DOI']]
df_ISGlobal['Center'] = 'ISGlobal'
df_ISGlobal

/tmp/ipykernel_20144/4287051607.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_ISGlobal['Center'] = 'ISGlobal'


,DOI,Center
0,10.1038/s41380-019-0558-2,ISGlobal
1,10.1016/j.edumed.2019.09.004,ISGlobal
2,10.1038/ng.2238,ISGlobal
3,10.1111/all.14422,ISGlobal
4,10.1016/j.nrl.2020.02.006,ISGlobal
...,...,...
753,10.3390/ijms222413363,ISGlobal
754,10.3390/pathogens10121588,ISGlobal
755,10.1371/journal.pntd.0009954,ISGlobal
756,10.1136/bmjopen-2021-052897,ISGlobal


### ResearchMar

In [16]:
center_name = 'ResearchMar/'
file_name = 'HMRIB_CERCA_DOIs21-24_VF'

In [17]:
df_ResearchMar = pd.read_excel(file_path + 'BM_' + center_name + file_name + '.xlsx', header = None)[0:-1].rename(columns = {0: 'DOI'})
df_ResearchMar['Center'] = 'ResearchMar'
df_ResearchMar

,DOI,Center
0,10.1001/jama.2021.15255,ResearchMar
1,10.1001/jama.2022.1645,ResearchMar
2,10.1001/jamacardio.2021.4690,ResearchMar
3,10.1001/jamacardio.2022.1988,ResearchMar
4,10.1001/jamadermatol.2022.0434,ResearchMar
...,...,...
5346,10.7759/cureus.13183,ResearchMar
5347,10.7759/cureus.16472,ResearchMar
5348,10.7759/cureus.40708,ResearchMar
5349,10.7759/cureus.62509,ResearchMar


In [18]:
df_centers = pd.concat([df_BETA, df_CREAF, df_ICN2, df_ISGlobal, df_ResearchMar], ignore_index = True)
df_centers.to_csv('../data/processed/CERCA_5_Bibliometria_SIRIS.csv', index = False)
df_centers

,DOI,Center
0,10.1016/j.scitotenv.2023.168824,BETA
1,10.23818/limn.43.07,BETA
2,10.1016/j.aquatox.2024.106843,BETA
3,10.3390/agronomy14050935,BETA
4,10.32347/2077-3455.2024.68.215-227,BETA
...,...,...
7933,10.7759/cureus.13183,ResearchMar
7934,10.7759/cureus.16472,ResearchMar
7935,10.7759/cureus.40708,ResearchMar
7936,10.7759/cureus.62509,ResearchMar


## Check which publications are not in OA using DOI

In [19]:
PROJECT_ID = 'siris-datasets'
DATASET_ID = 'openalex'

def bg_query(query):
    client = bigquery.Client(project=PROJECT_ID)
    df = client.query(query)
    return df.to_dataframe()

In [20]:
in_query = str(tuple(df_centers.DOI.tolist()))
in_query = in_query.replace(',)', ')')

sql = f"""SELECT ww.DOI,
       a.display_name,
       wa.author_order,
       wins.ID AS institution_id,
       wins.COUNTRY_CODE
      FROM `{PROJECT_ID}.{DATASET_ID}.works` ww
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.works_authorships` wa ON wa.WORK_ID = ww.ID
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.institutions` wins ON wins.ID = wa.INSTITUTION_ID
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.authors` a ON a.ID = wa.author_id
      WHERE ww.DOI IN {in_query}
      """
df_OA = bg_query(sql).dropna(subset = 'DOI').reset_index(drop = True)
df_OA

/home/siris/2025CERCA01/2025CERCA01_env/lib/python3.10/site-packages/google/auth/_default.py:108: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
/home/siris/2025CERCA01/2025CERCA01_env/lib/python3.10/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,DOI,display_name,author_order,institution_id,COUNTRY_CODE
0,10.1002/wps.20971,David R. Williams,58,136199984,US
1,10.1016/j.jaad.2020.10.046,Alexandra Perez Mariño,74,<NA>,None
2,10.1111/all.15679,R. Emuzyte,141,173212132,LT
3,10.1111/all.15679,Thomas Eiwegger,139,4210141030,CA
4,10.1111/all.15679,Thomas Eiwegger,139,2801317318,CA
...,...,...,...,...,...
189137,10.1002/clt2.12062,Nikolaos G. Papadopoulos,33,200777214,GR
189138,10.1002/clt2.12062,Nikolaos G. Papadopoulos,33,28407311,GB
189139,10.1016/s2468-2667(21)00065-7,Cătălina Liliana Andrei,33,<NA>,None
189140,10.1007/s00464-024-11109-x,J Guevara-Martínez,33,2800562746,ES


In [21]:
(df_OA.DOI.nunique()) / df_centers.DOI.nunique()

0.9305475504322767

**Lets try to see if we can get them from Open aire**

In [ ]:
import requests
import pandas as pd
from tqdm import tqdm
import time
from concurrent.futures import ThreadPoolExecutor, as_completed, TimeoutError

def extract_field(field):
    """Extract the first value of a list or $ value from a dict."""
    if isinstance(field, list):
        return field[0].get("$", None) if field else None
    elif isinstance(field, dict):
        return field.get("$", None)
    return field

BATCH_SIZE = 25
SLEEP_BETWEEN_REQUESTS = 1.0
BASE_URL = "https://api.openaire.eu/search/publications"
TIMEOUT_PER_DOI = 30           # max seconds per DOI request
TIMEOUT_PER_RESULT = 30        # max seconds per result processing

records_list = []
not_found_dois = []

df_missing = df_centers[~df_centers.DOI.isin(df_OA.DOI.unique())]

def process_single_result(result):
    """Process a single OpenAIRE result, extracting author-affiliation rows."""
    metadata = result.get("metadata", {}).get("oaf:entity", {}).get("oaf:result", {})
    doi_found = extract_field(metadata.get("pid"))
    authors = metadata.get("creator", [])
    rels = metadata.get("rels", {}).get("rel", [])
    doi_records = []

    for idx, author in enumerate(authors, start=1):
        author_firstname = extract_field(author.get("@name"))
        author_surname = extract_field(author.get("@surname"))
        author_name = f"{author_firstname} {author_surname}" if author_firstname or author_surname else None
        author_order = extract_field(author.get("@rank")) or idx

        author_rels = [rel for rel in rels if rel.get("to", {}).get("@class") == "hasAuthorInstitution"]
        if author_rels:
            for rel in author_rels:
                institution_id = rel["to"].get("$")
                country = rel.get("country", {}).get("@classname")
                doi_records.append({
                    "DOI": doi_found,
                    "author_name": author_name,
                    "author_order": author_order,
                    "institution_id": institution_id,
                    "COUNTRY_CODE": country
                })
        else:
            doi_records.append({
                "DOI": doi_found,
                "author_name": author_name,
                "author_order": author_order,
                "institution_id": None,
                "COUNTRY_CODE": None
            })

    return doi_records

# Process DOIs batch by batch
for i in tqdm(range(0, len(df_missing.DOI), BATCH_SIZE)):
    batch = df_missing.DOI[i:i+BATCH_SIZE]
    
    for doi in batch:
        try:
            # Fetch DOI with timeout
            with ThreadPoolExecutor(max_workers=1) as executor:
                future = executor.submit(
                    requests.get,
                    BASE_URL,
                    params={"doi": doi, "format": "json", "size": 1},
                    timeout=30
                )
                try:
                    response = future.result(timeout=TIMEOUT_PER_DOI)
                    response.raise_for_status()
                    openaire_results = response.json()
                except TimeoutError:
                    print(f"⏱ Timeout fetching DOI {doi}, skipping.")
                    not_found_dois.append(doi)
                    continue

            results = openaire_results.get("response", {}).get("results", {}).get("result", [])
            if not results:
                not_found_dois.append(doi)
                continue
            if not isinstance(results, list):
                results = [results]

            # Process each result with its own timeout
            for result in results:
                with ThreadPoolExecutor(max_workers=1) as executor:
                    future = executor.submit(process_single_result, result)
                    try:
                        doi_records = future.result(timeout=TIMEOUT_PER_RESULT)
                        if doi_records:
                            records_list.extend(doi_records)
                    except TimeoutError:
                        print(f"⏱ Timeout processing a result for DOI {doi}, skipping this result.")

            time.sleep(SLEEP_BETWEEN_REQUESTS)

        except Exception as e:
            print(f"⚠️ Error processing DOI {doi}: {e}")
            not_found_dois.append(doi)
            

# Convert list to DataFrame once at the end
records = pd.DataFrame(records_list)
print("Number of DOIs not found:", len(set(not_found_dois)))
records = records.drop_duplicates(['DOI', 'institution_id', 'author_name']).reset_index(drop = True)
records


  0%|          | 0/23 [00:00<?, ?it/s]

⚠️ Error processing DOI 10.3390/su16010238.: 'NoneType' object has no attribute 'get'
⚠️ Error processing DOI 10.5683/SP3/BIDMCI: 'NoneType' object has no attribute 'get'
⚠️ Error processing DOI 10.1109/MetroAgriFor63043.2024.10948835: 'str' object has no attribute 'get'
⚠️ Error processing DOI 10.1017/S0007485324000567: 'str' object has no attribute 'get'
⚠️ Error processing DOI 10.70186/baeeZTOI5976: 'NoneType' object has no attribute 'get'
⚠️ Error processing DOI 10.21138/GF.830: 'str' object has no attribute 'get'


 13%|█▎        | 3/23 [01:48<12:23, 37.16s/it]

⚠️ Error processing DOI 10.1098/rstb.2019.0810rstb20190810: 'NoneType' object has no attribute 'get'
⚠️ Error processing DOI 10.1016/S2352-3018(21)00051-5 : 'NoneType' object has no attribute 'get'


 17%|█▋        | 4/23 [02:22<11:25, 36.06s/it]

...

 87%|████████▋ | 20/23 [12:14<01:50, 36.72s/it]

⚠️ Error processing DOI 10.23736/S1973-9087.24.08234-0: 'str' object has no attribute 'get'
⚠️ Error processing DOI 10.25259/IJDVL_561_2022: 'str' object has no attribute 'get'
⚠️ Error processing DOI 10.25259/IJDVL_561_2022: 'str' object has no attribute 'get'


100%|██████████| 23/23 [13:51<00:00, 36.13s/it]


Number of DOIs not found: 25


,DOI,author_name,author_order,institution_id,COUNTRY_CODE
0,10.7818/ecos.2684,Meritxell Abril,1,openorgs____::6c3442fa797102c1c5c1a4319fce4119,Spain
1,10.7818/ecos.2684,Meritxell Abril,1,openorgs____::da01441f0598813c643e481cd22673d5,Spain
2,10.7818/ecos.2684,Isabel Muñoz,2,openorgs____::6c3442fa797102c1c5c1a4319fce4119,Spain
3,10.7818/ecos.2684,Isabel Muñoz,2,openorgs____::da01441f0598813c643e481cd22673d5,Spain
4,10.7818/ecos.2684,Margarita Menéndez,3,openorgs____::6c3442fa797102c1c5c1a4319fce4119,Spain
...,...,...,...,...,...
782759,10.7554/elife.85893,Bernardo Rudy,11,openorgs____::e1bc2739ac515ca512dc9e814cad9613,Italy
782760,10.7554/elife.85893,Bernardo Rudy,11,openorgs____::38dea46ca400398cccbaf76ca0ba85c2,United States
782761,10.7554/elife.85893,Bernardo Rudy,11,openorgs____::e08612ddafc9e1533bb1c03b7a550a87,United States
782762,10.7554/elife.85893,Bernardo Rudy,11,openorgs____::d41cf6bd4ab1b1362a44397e0b95c975,Italy


In [ ]:
df_DOI = pd.concat([df_OA, records.rename(columns = {'author_name' : 'display_name'})], ignore_index = True)
df_DOI.to_csv('../data/interim/CERCA_5_Bibliometria_SIRIS_processed.csv', index = False)
df_DOI

,DOI,display_name,author_order,institution_id,COUNTRY_CODE
0,10.1093/bjs/znab247,David Merlini,153,<NA>,None
1,10.1093/bjs/znab247,U. Novo Rivas,642,<NA>,None
2,10.1093/bjs/znab247,Basim Al-Khafaji,103,<NA>,None
3,10.1093/bjs/znab247,Vincenzo Papagni,663,<NA>,None
4,10.1038/s41467-024-49494-5,Edurne Martínez del Castillo,39,197323543,DE
...,...,...,...,...,...
634027,10.7554/elife.85893,Bernardo Rudy,11,openorgs____::e1bc2739ac515ca512dc9e814cad9613,Italy
634028,10.7554/elife.85893,Bernardo Rudy,11,openorgs____::38dea46ca400398cccbaf76ca0ba85c2,United States
634029,10.7554/elife.85893,Bernardo Rudy,11,openorgs____::e08612ddafc9e1533bb1c03b7a550a87,United States
634030,10.7554/elife.85893,Bernardo Rudy,11,openorgs____::d41cf6bd4ab1b1362a44397e0b95c975,Italy


In [ ]:
df_DOI[df_DOI.COUNTRY_CODE.isin(['ES', 'Spain'])].drop_duplicates(['display_name', 'institution_id']).dropna()

,DOI,display_name,author_order,institution_id,COUNTRY_CODE
5,10.1016/j.euroneuro.2023.06.003,Elena Rubio‐Abadal,37,4210127203,ES
7,10.1016/j.euroneuro.2023.06.003,Elena Rubio‐Abadal,37,4210128981,ES
10,10.1016/j.clgc.2022.06.001,Alejo Rodríguez‐Vida,35,4210130874,ES
27,10.3390/jcm10194402,J L Rueda García,44,2800562746,ES
32,10.1111/apt.18133,Raquel Mena,38,4210098336,ES
...,...,...,...,...,...
633962,10.7554/elife.81067,Study Oasis,31,pending_org_::97fcd65a2bc9fd4448e94200efe6a7ba,Spain
633964,10.7554/elife.81067,Study Oasis,31,openorgs____::08b006c8328e8d0320d28e35ed34b125,Spain
633967,10.7554/elife.81067,Study Oasis,31,cf__________::1170febd100770dd345d6ed983bf6c48,Spain
633968,10.7554/elife.81067,Study Oasis,31,pending_org_::04c48928af1d147d20064809e815b171,Spain


### Identify CERCA authors using OA (93% of the dataset)

- By affiliation ID
- By raw affiliations
  - Using parents affiliations
  - Manually checking above > 1 per raw affiliation [2k affiliations]
  - String search with keywords for = 1 doi per raw affilation [4k affilations]

**We identify 70% of the provided DOI's**

For the ResearchMar the retrieval is 61%; being the center with most publications we will force the manual identification to increase it by selecting in a second phase using a manual validation [word mar and Barcelona]. We reach 67%

The problem is the hypothesis of using the parent affiliations (problem for the hospital) that is not good enough but I can't manually review all the affiliations of spain. So I can't improve it further

So to solve it we select the not found dois in Researchmar using the previous parent affiliations hypothesis and chec the raw affiliations in Spain looking for strings (mar, imim, parc combined with barcelona without checking).We reach 84%

In [22]:
cerca_centers = {'BETA' : [''], # NOT IN OA
                    'CREAF' : ['4210129656', '4401200259'],
                    'ICN2' : ['4210093216'],
                    'ISGlobal' : ['4210148332'],
                    'ResearchMar' : ['4210156109']}

interest_cerca = list(set(sum(list(cerca_centers.values()), [])))
interest_cerca = [int(x) for x in interest_cerca if x != '']

cerca_authors = df_OA[df_OA.institution_id.isin(interest_cerca)].drop_duplicates(['DOI'])

cerca_parents = {'BETA' : ['115304662'],
                    'CREAF' : ['123044942', '71999127'],
                    'ICN2' : ['123044942'],
                    'ISGlobal' : ['123044942', '170486558'],
                    'ResearchMar' : ['170486558']}

parents_cerca = list(set(sum(list(cerca_parents.values()), [])))
parents_cerca = [int(x) for x in parents_cerca if x != '']

cerca_possible_authors = df_OA[(df_OA.institution_id.isin(parents_cerca)) & (~df_OA.display_name.isin(cerca_authors.display_name))].drop_duplicates(['DOI'])
cerca_possible_authors

,DOI,display_name,author_order,institution_id,COUNTRY_CODE
438,10.1016/j.jamda.2020.09.032,Alessandra Coin,169,123044942,ES
753,10.1038/s41375-022-01802-y,Bárbara Tazón‐Vega,45,123044942,ES
1015,10.1093/jac/dkab069,Carlos Falces,38,71999127,ES
1277,10.1007/s00134-023-07161-1,Anna Pous,224,71999127,ES
1305,10.1186/s12889-022-13724-6,Pere Ginès,47,71999127,ES
...,...,...,...,...,...
184005,10.1016/j.ad.2024.03.024,Elena Giménez‐Arnau,29,170486558,ES
184561,10.1016/j.jaip.2024.03.050,Sami Aqel,29,123044942,ES
184986,10.1136/ard-2023-224990,Núria Guañabens,29,71999127,ES
186543,10.5853/jos.2021.00962,Josep Maria Aragonés,31,115304662,ES


In [23]:
in_query = str(tuple(cerca_possible_authors.DOI.tolist()))
in_query = in_query.replace(',)', ')')

sql = f"""SELECT ww.DOI,
       a.display_name,
       war.raw_affiliation,
       wins.ID AS institution_id,
       wins.COUNTRY_CODE
      FROM `{PROJECT_ID}.{DATASET_ID}.works` ww
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.works_authorships` wa ON wa.WORK_ID = ww.ID
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.institutions` wins ON wins.ID = wa.INSTITUTION_ID
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.authors` a ON a.ID = wa.author_id
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.works_authorships_raw` war ON war.WORK_ID = ww.ID AND war.AUTHOR_ID = wa.AUTHOR_ID
      WHERE ww.DOI IN {in_query}
      """
df_possible_cerca = bg_query(sql).dropna(subset = 'DOI').drop_duplicates(['DOI', 'raw_affiliation']).reset_index(drop = True).reset_index(drop = True)
df_possible_cerca

/home/siris/2025CERCA01/2025CERCA01_env/lib/python3.10/site-packages/google/auth/_default.py:108: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
/home/siris/2025CERCA01/2025CERCA01_env/lib/python3.10/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,DOI,display_name,raw_affiliation,institution_id,COUNTRY_CODE
0,10.1016/j.parkreldis.2024.106993,Ana Cámara,Parkinson's Disease & Movement Disorders Unit ...,71999127,ES
1,10.1002/art.41999,Jordi Yagüe,"Hospital Clínic de Barcelona, Institut d'Inves...",71999127,ES
2,10.1016/j.cireng.2022.09.016,Miguel Pera-Román,"Unidad de Cirugía de Urgencias, Sección de Cir...",123044942,ES
3,10.1016/j.cireng.2022.09.016,Miguel Pera-Román,"Departamento de Cirugía, Universidad Autónoma ...",123044942,ES
4,10.1016/j.clgc.2022.08.006,Giuseppe Di Lorenzo,"Oncology University of Molise and ASL, Salerno...",129627893,IT
...,...,...,...,...,...
32227,10.1016/j.medine.2022.12.002,Mercedes Catalán-González,"Service of Intensive Care Medicine, Hospital 1...",4210086614,ES
32228,10.3324/haematol.2023.283209,Marco M. Bühler,Department of Pathology and Molecular Patholog...,4210100468,CH
32229,10.1038/s41598-021-97140-7,Urban Ösby,"Department of Neurobiology, Care Sciences, and...",28166907,SE
32230,10.1038/s41375-022-01802-y,Lars Bullinger,"Department of Hematology, Oncology and Cancer ...",75951250,DE


In [24]:
grouped = df_possible_cerca[df_possible_cerca.COUNTRY_CODE == 'ES'].groupby('raw_affiliation').count().sort_values('DOI', ascending = False)[['DOI']]
to_check = grouped[grouped.DOI > 1]
print(df_possible_cerca[df_possible_cerca.raw_affiliation.isin(to_check.index)].DOI.nunique())
# to_check.to_csv('to_check.csv')
to_check
### WE MISS 876 DOIS BY FILTERING CHECKING THE THRESHOLD OF 1 AND MANUALLY CHECK THE MISSING

1533


,DOI
raw_affiliation,
"Universitat Pompeu Fabra (UPF), Barcelona, Spain",68
"IMIM (Hospital del Mar Medical Research Institute), Barcelona, Spain",57
"ISGlobal, Barcelona, Spain",48
"Universitat Pompeu Fabra, Barcelona, Spain",41
"Hospital del Mar Medical Research Institute (IMIM), Barcelona, Spain",40
...,...
"Department of Dermatology, Hospital Universitari Arnau de Vilanova - Institut de Recerca Biomèdica de Lleida, Lleida, Spain",2
"Department of Dermatology, Hospital Universitario Virgen de la Victoria, Málaga, Spain",2
"Lipid Unit, Department of Internal Medicine, IDIBELL-Hospital Universitari de Bellvitge, L’Hospitalet de Llobregat, FIPEC, 08908 Barcelona, Spain",2


In [31]:
df_check = g2d.download('1DtbOJzE9c9xCtuMfQQX8f4Uom6U7Mx7wwmzri57Xj4A', '>1', col_names = True, row_names = False)

compute  = df_possible_cerca[df_possible_cerca.raw_affiliation.isin(df_check[df_check.CERCA == 'TRUE'].raw_affiliation)].DOI.nunique()

(compute + cerca_authors.DOI.nunique()) / df_centers.DOI.nunique()

Not all requested scopes were granted by the authorization server, missing scopes https://docs.google.com/feeds, https://spreadsheets.google.com/feeds.


0.6455331412103746

In [26]:
cerca_string = {'BETA' : ['vic', 'beta'], # NOT IN OA
                    'CREAF' : ['creaf', 'cerdanyola', 'bellaterra'],
                    'ICN2' : ['icn2', 'bellaterra', 'cerdanyola'],
                    'ISGlobal' : ['global'],
                    'ResearchMar' : ['mar', 'imim']}
cerca_string = list(set(sum(list(cerca_string.values()), [])))


df_tmp = grouped[(grouped.DOI == 1) & (~grouped.index.isin(df_check.raw_affiliation))].reset_index()
df_tmp['raw_affiliation'] = df_tmp['raw_affiliation'].str.lower()

df_tmp_1 = df_tmp[df_tmp['raw_affiliation'].str.contains(('|'.join(cerca_string)), case=False, na=False)].drop_duplicates().reset_index(drop = True)
# df_tmp_1.to_csv('to_check_v2.csv')
df_tmp_1

,raw_affiliation,DOI
0,instituto de investigación biosanitaria de gra...,1
1,institute of biomedical research of salamanca ...,1
2,instituto murciano de investigación biosanitar...,1
3,instituto murciano de investigación biosanitar...,1
4,instituto hospital del mar de investigación mé...,1
...,...,...
4502,"department of neurology, neurovascular researc...",1
4503,"department of neurology, neurovascular researc...",1
4504,"department of neuropsychiatry and addictions, ...",1
4505,"department of neuropsychiatry and addictions, ...",1


In [75]:
df_check_v2 = g2d.download('1DtbOJzE9c9xCtuMfQQX8f4Uom6U7Mx7wwmzri57Xj4A', '=1', col_names = True, row_names = False)
df_check_v2

Not all requested scopes were granted by the authorization server, missing scopes https://docs.google.com/feeds, https://spreadsheets.google.com/feeds.


,raw_affiliation,DOI,CERCA,CERCA_IMIM
0,". clínica de rehabilitación de salud mental, d...",1,FALSE,
1,. department of child and adolescent psychiatr...,1,FALSE,FALSE
2,". department of psychiatry, chinese university...",1,FALSE,FALSE
3,. hospital del mar medical research institute ...,1,TRUE,
4,". infectious diseases service, hospital del ma...",1,TRUE,
...,...,...,...,...
4543,wildlife ecology & health group (we&h) and ser...,1,FALSE,
4544,"wildlife ecology & health group (we&h), and se...",1,FALSE,
4545,"wildlife ecology & health group (we&h), servei...",1,FALSE,
4546,wildlife ecology & health research group (we&h...,1,FALSE,


In [76]:
check_1 = df_possible_cerca[df_possible_cerca.raw_affiliation.isin(df_check[df_check.CERCA == 'TRUE'].raw_affiliation)]
check_1_imim = df_possible_cerca[df_possible_cerca.raw_affiliation.isin(df_check[df_check.CERCA_IMIM == 'TRUE'].raw_affiliation)]
check_2 = df_possible_cerca[df_possible_cerca.raw_affiliation.str.lower().isin(df_check_v2[df_check_v2.CERCA == 'TRUE'].raw_affiliation)]
check_2_imim = df_possible_cerca[df_possible_cerca.raw_affiliation.str.lower().isin(df_check_v2[df_check_v2.CERCA_IMIM == 'TRUE'].raw_affiliation)]

cerca_doi = list(set(
    list(cerca_authors.DOI.unique()) +
    # list(cerca_possible_authors.DOI.unique()) +
    list(check_1.DOI.unique()) +
    list(check_2.DOI.unique()) +
    list(check_1_imim.DOI.unique()) +
    list(check_2_imim.DOI.unique())
))
len(cerca_doi) / df_centers.DOI.nunique()

0.6811239193083574

In [84]:
cerca_possible_authors_not_parent = df_ResearchMar[~df_ResearchMar.DOI.isin(cerca_doi)]

in_query = str(tuple(cerca_possible_authors_not_parent.DOI.tolist()))
in_query = in_query.replace(',)', ')')

sql = f"""SELECT ww.DOI,
       a.display_name,
       war.raw_affiliation,
       wins.ID AS institution_id,
       wins.COUNTRY_CODE
      FROM `{PROJECT_ID}.{DATASET_ID}.works` ww
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.works_authorships` wa ON wa.WORK_ID = ww.ID
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.institutions` wins ON wins.ID = wa.INSTITUTION_ID
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.authors` a ON a.ID = wa.author_id
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.works_authorships_raw` war ON war.WORK_ID = ww.ID AND war.AUTHOR_ID = wa.AUTHOR_ID
      WHERE ww.DOI IN {in_query}
      """
df_possible_cerca_not_parent = bg_query(sql).dropna(subset = 'DOI').drop_duplicates(['DOI', 'raw_affiliation']).reset_index(drop = True).reset_index(drop = True)
df_possible_cerca_not_parent

/home/siris/2025CERCA01/2025CERCA01_env/lib/python3.10/site-packages/google/auth/_default.py:108: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
/home/siris/2025CERCA01/2025CERCA01_env/lib/python3.10/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,DOI,display_name,raw_affiliation,institution_id,COUNTRY_CODE
0,10.1016/j.ejso.2020.12.006,Richa Patel,"Auckland City Hospital, New Zealand",2801418622,NZ
1,10.2217/fon-2020-0935,Kōichi Goto,"National Cancer Center Hospital East, Chiba, J...",4210145079,JP
2,10.1016/j.thromres.2021.06.008,Koh Ono,"Department of Cardiovascular Medicine, Graduat...",22299242,JP
3,10.1016/j.waojou.2021.100542,Kiran Godse,"Department of Dermatology, D Y, Patil Universi...",4210153931,IN
4,10.1093/brain/awab362,Liisa Metsähonkala,"Epilepsy Unit, Hospital for Children and Adole...",4210090554,FI
...,...,...,...,...,...
17625,10.1016/j.ejca.2024.113530,Joaquim Bellmunt,"Dana-Farber Cancer Institute and IMIM Lab, 450...",4210117453,US
17626,10.1186/s12916-024-03719-y,Michael I. Goran,"Division of Endocrinology, Diabetes and Metabo...",1336910626,US
17627,10.1093/jac/dkae363,Miriam O’Hare,"Micron Research Ltd , 109B Lancaster Way, Ely ...",11912373,US
17628,10.1016/j.jhazmat.2022.128563,Natàlia García‐Reyero,"Environmental Laboratory, US Army Engineer Res...",87303767,US


In [88]:
grouped = df_possible_cerca_not_parent[df_possible_cerca_not_parent.COUNTRY_CODE == 'ES'].groupby('raw_affiliation').count().sort_values('DOI', ascending = False)[['DOI']]
to_check = grouped[grouped.DOI > 0]
to_check.to_csv( 'to_check_v3.csv' )
to_check

,DOI
raw_affiliation,
"Hospital del Mar, Barcelona, Spain",41
"Department of Dermatology, Hospital del Mar, Barcelona, Spain",13
"Hospital Universitario 12 de Octubre, Madrid, Spain",13
"Hospital Universitario Central de Asturias, Oviedo, Spain",12
"Medical Oncology Department, Hospital del Mar, Barcelona, Spain",11
...,...
"Department of Surgery, Hospital Universitario Mollet, Mollet, 08100, Spain",1
"Department of Surgery, Hospital Universitario Miguel Servet, Zaragoza, Spain.",1
"Department of Surgery, Hospital Universitario Miguel Servet, Zaragoza, Spain",1


In [89]:
df_check_v3 = g2d.download('1DtbOJzE9c9xCtuMfQQX8f4Uom6U7Mx7wwmzri57Xj4A', 'Nonparent', col_names = True, row_names = False)
df_check_v3

Not all requested scopes were granted by the authorization server, missing scopes https://docs.google.com/feeds, https://spreadsheets.google.com/feeds.


,raw_affiliation,DOI,CERCA
0,"Hospital del Mar, Barcelona, Spain",41,TRUE
1,"Department of Dermatology, Hospital del Mar, B...",13,TRUE
2,"Hospital Universitario 12 de Octubre, Madrid, ...",13,FALSE
3,"Hospital Universitario Central de Asturias, Ov...",12,FALSE
4,"Medical Oncology Department, Hospital del Mar,...",11,TRUE
...,...,...,...
8398,"Department of Surgery, Hospital Universitario ...",1,FALSE
8399,"Department of Surgery, Hospital Universitario ...",1,FALSE
8400,"Department of Surgery, Hospital Universitario ...",1,FALSE
8401,"Department of Surgery, Hospital Universitario ...",1,FALSE


In [99]:
check_1 = df_possible_cerca[df_possible_cerca.raw_affiliation.isin(df_check[df_check.CERCA == 'TRUE'].raw_affiliation)]
check_1_imim = df_possible_cerca[df_possible_cerca.raw_affiliation.isin(df_check[df_check.CERCA_IMIM == 'TRUE'].raw_affiliation)]
check_2 = df_possible_cerca[df_possible_cerca.raw_affiliation.str.lower().isin(df_check_v2[df_check_v2.CERCA == 'TRUE'].raw_affiliation)]
check_2_imim = df_possible_cerca[df_possible_cerca.raw_affiliation.str.lower().isin(df_check_v2[df_check_v2.CERCA_IMIM == 'TRUE'].raw_affiliation)]
check_3 = df_possible_cerca_not_parent[df_possible_cerca_not_parent.raw_affiliation.isin(df_check_v3[df_check_v3.CERCA == 'TRUE'].raw_affiliation)]


cerca_doi = list(set(
    list(cerca_authors.DOI.unique()) +
    # list(cerca_possible_authors.DOI.unique()) +
    list(check_1.DOI.unique()) +
    list(check_2.DOI.unique()) +
    list(check_1_imim.DOI.unique()) +
    list(check_2_imim.DOI.unique()) + 
    list(check_3.DOI.unique())
))
len(cerca_doi) / df_centers.DOI.nunique()

0.8435158501440922

In [112]:
df_check = pd.concat((cerca_authors, check_1, check_2, check_1_imim, check_2_imim, check_3))
df_check

,DOI,display_name,author_order,institution_id,COUNTRY_CODE,raw_affiliation
116,10.1159/000513538,Josep M. Anto,39,4210148332,ES,NaN
414,10.1016/j.gim.2022.06.011,Ana Pozueta,58,4210156109,ES,NaN
540,10.3389/fmed.2020.607786,Clara Barrios,50,4210156109,ES,NaN
1054,10.1038/s41588-021-00936-6,Aida Peiró-Mestres,168,4210148332,ES,NaN
1732,10.1016/j.jval.2021.03.021,Juan Moróte,44,4210156109,ES,NaN
...,...,...,...,...,...,...
14225,10.1515/almed-2024-0164,Javier Hernando Redondo,<NA>,4210130874,ES,"Hospital del Mar de Barcelona , Barcelona , Spain"
14232,10.1007/s41999-023-00875-x,Anna Renom‐Guiteras,<NA>,4210133106,ES,"Department of Geriatric Medicine, Parc de Salu..."
14249,10.1007/s10815-020-01934-z,Miguel Ángel Checa,<NA>,4210133106,ES,"Department of Obstetrics and Gynecology, Parc ..."
14255,10.1080/14779072.2024.2349103,Teresa Guiberteau-Diaz,<NA>,4210130874,ES,"Cardiology Department, Hospital del Mar, Barce..."


In [122]:
# HI HA ERROR AMB L'ASSIGNACIÓ I PER AIXÒ SURT RAR! S'HA DE FER ALS POSSIBLE (PATENT I NO PARENT) I LLAVORS FER EL MERGE AMB EL OA QUE TÉ L'AUTHOR ORDER

df_OA['CERCA'] = (df_OA['display_name'].isin(df_check['display_name']) & df_OA['DOI'].isin(df_check['DOI']))
df_final = df_OA.merge(df_centers, on = 'DOI').drop_duplicates().reset_index(drop = True)
df_final.to_csv('../data/processed/CERCA_5_Bibliometria_SIRIS_OA.csv', index = False)
df_final

,DOI,display_name,author_order,institution_id,COUNTRY_CODE,CERCA,Center
0,10.1002/wps.20971,David R. Williams,58,136199984,US,False,ResearchMar
1,10.1016/j.jaad.2020.10.046,Alexandra Perez Mariño,74,<NA>,None,False,ResearchMar
2,10.1111/all.15679,R. Emuzyte,141,173212132,LT,False,ResearchMar
3,10.1111/all.15679,Thomas Eiwegger,139,4210141030,CA,False,ResearchMar
4,10.1111/all.15679,Thomas Eiwegger,139,2801317318,CA,False,ResearchMar
...,...,...,...,...,...,...,...
193398,10.1002/clt2.12062,Nikolaos G. Papadopoulos,33,28407311,GB,False,ISGlobal
193399,10.1002/clt2.12062,Nikolaos G. Papadopoulos,33,28407311,GB,False,ResearchMar
193400,10.1016/s2468-2667(21)00065-7,Cătălina Liliana Andrei,33,<NA>,None,False,ISGlobal
193401,10.1007/s00464-024-11109-x,J Guevara-Martínez,33,2800562746,ES,False,ResearchMar


In [123]:
df_final[df_final.CERCA == True].drop_duplicates(['DOI', 'Center']).groupby('Center').size()

Center
BETA             50
CREAF           825
ICN2            712
ISGlobal        626
ResearchMar    3725
dtype: int64